In [12]:
import sys, os
import pandas as pd
from utils.misc import cols_to_front
from pathlib import Path
from datetime import datetime
import shutil
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor
import yaml
with open('config/config.yaml', 'r') as file:
    config = yaml.safe_load(file)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

In [13]:
run_name = (
    f"{config.get('exp_name','NA').upper()}"
    f"_rd-{str(config.get('reduce_dim','NA')).lower()}"
    f"_meth-{config.get('reduce_dim_method','NA')}"
    f"_dim-{config.get('n_dim','NA')}"
    f"_idf-{str(config.get('idf','NA')).lower()}"
)

# Add timestamp for uniqueness
run_name = f"{run_name}_{timestamp}"

# Run directory
RUN_DIR = Path(config["run_dir"]) / run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(RUN_DIR)

runs\FLAVOR_MATCH_rd-true_meth-svd_dim-100_idf-false_2026-03-05_12-04


# Run the pipeline

In [14]:
# Run all the pipeline notebooks sequentially
NOTEBOOK_DIR = Path(".")
notebooks = [
    "data_processing.ipynb",
    'vectorize_ai_descriptors.ipynb',
    'ingredients_matching.ipynb',
    'data_exploration.ipynb',
]

# -----------------------------------
# Execute notebooks sequentially
# -----------------------------------
for nb_name in notebooks:
    nb_path = NOTEBOOK_DIR / nb_name
    print(f"\n Executing {nb_name}")

    with open(nb_path) as f:
        nb = nbformat.read(f, as_version=4)

    ep = ExecutePreprocessor(
        timeout=600,          # increase if long runs
        kernel_name="python3"
    )

    ep.preprocess(nb, {"metadata": {"path": NOTEBOOK_DIR}})

    # Overwrite notebook with executed outputs
    with open(nb_path, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"✅ Finished {nb_name}")

# Saving files

In [15]:
data_files = [
    'config/config.yaml',
    'data/merged_ai_descriptors_clean.csv',
    'data/local_df_ai_descriptors.csv',
    'data/merged_ai_descriptors_dummies_filtered.csv',
    'data/merged_ai_descriptors_dummies_full.csv',
    'data/merged_ai_descriptors_clean.csv',
    'data/merged_ai_descriptors.csv',
    'data/flavor_db_descriptors.csv',
    'data/local_df_ai_descriptors.csv',
    'data/all_descriptors.csv'
]

output_files = [
    'output/flavordb_ingredients_top3_matches.xlsx',
    'output/local_ingredients_top3_matches.xlsx',
    'output/cat_clustered_heatmap.png',
    'output/db_jaccard.png',
    'output/cat_tsne.png',
    'output/db_tsne.png',
    "output/category_similarity_top3.xlsx",
    "output/local_flavordb_similarity.xlsx",
    "output/local_flavordb_general_similarity.xlsx"
    
]

In [16]:
# Save data files
data_dir = RUN_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in data_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

In [17]:
# Save output files
data_dir = RUN_DIR / "output"
data_dir.mkdir(parents=True, exist_ok=True)

# copy files
for file in output_files:
    src = Path(file)
    dst = data_dir / src.name
    shutil.copy2(src, dst)

# Save overview

In [18]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------
# Input files (your lists)
# -----------------------------
xlsx_files = [
    'output/local_ingredients_top3_matches.xlsx',
    'output/flavordb_ingredients_top3_matches.xlsx'
]

png_files = [
    'output/cat_tsne.png',
    'output/cat_clustered_heatmap.png',
]

cols_to_drop = ["raw_AI_output", "sub_category", "db", "category"]

df1 = pd.read_excel(xlsx_files[0])
df2 = pd.read_excel(xlsx_files[1])

# Drop unwanted columns
df1 = df1.drop(columns=cols_to_drop, errors="ignore")
df2 = df2.drop(columns=cols_to_drop, errors="ignore")

# Selection of rows
ings =['Macrocystis pyrifera', 'Morchella crassipes', 'Picea rubens', 'Allium canadense', 'Rhus glabra', 'Artemisia frigida']
df1 = df1[df1["Nom scientifique"].isin(ings)]   

ings = ['Rice', 'Cedar', 'Sage', 'Apple', "Celery", "Lemon"]
df2 = df2[df2["Nom scientifique"].isin(ings)]   

In [19]:
# Rename matches to common name
merged_df = pd.read_csv('data/local_df_ai_descriptors.csv')

# Create mapping dict
mapping = dict(zip(
    merged_df["Nom scientifique"],
    merged_df["Nom vernaculaire"]
))

# Apply mapping (preserve original if no match)
cols = ["top1_name", "top2_name", "top3_name"]

for col in cols:
    df2[col] = df2[col].map(mapping)

# Combine into one comma-separated column
df2["matches"] = (
    df2[cols]
    .astype(str)
    .apply(lambda row: ", ".join(row.dropna()), axis=1)
)

# Drop some columns to make things more readable
df1 = df1[['Nom scientifique', 'Nom vernaculaire', 'matches', 'descriptor']]
df2 = df2[['Nom scientifique', 'matches', 'descriptor']]


In [20]:
from PIL import Image, ImageDraw, ImageFont
import textwrap

def make_header_image(config, width, dpi=300, pad_x=60, pad_y=30, bg=(255, 255, 255), fg=(0, 0, 0)):
    # Build the header text (use config values; fall back to your requested defaults if missing)
    exp_name = config.get("exp_name", "NA")
    reduce_dim = config.get("reduce_dim", "NA")
    reduce_dim_method = config.get("reduce_dim_method", "NA")
    n_dim = config.get("n_dim", "NA")
    idf = config.get("idf", "NA")

    header_text = (
        f'exp_name: "{exp_name}"    '
        f"reduce_dim: {str(reduce_dim).lower()}    "
        f'reduce_dim_method: "{reduce_dim_method}"    '
        f"n_dim: {n_dim}    "
        f"idf: {str(idf).lower()}"
    )

    # Font: try a nicer TTF, fallback to default
    try:
        # Common on many systems; replace if you have a specific font path
        font = ImageFont.truetype("arial.ttf", size=120)
    except Exception:
        font = ImageFont.load_default()

    # Wrap if needed
    draw_dummy = ImageDraw.Draw(Image.new("RGB", (width, 10), bg))
    max_chars = max(20, int(width / 18))  # heuristic
    lines = textwrap.wrap(header_text, width=max_chars) or [header_text]

    # Measure height
    line_heights = []
    line_widths = []
    for line in lines:
        bbox = draw_dummy.textbbox((0, 0), line, font=font)
        line_widths.append(bbox[2] - bbox[0])
        line_heights.append(bbox[3] - bbox[1])

    text_h = sum(line_heights) + int((len(lines) - 1) * (pad_y * 0.4))
    header_h = pad_y * 2 + text_h

    header = Image.new("RGB", (width, header_h), bg)
    draw = ImageDraw.Draw(header)

    # Optional: subtle separator line at bottom
    draw.line([(0, header_h - 2), (width, header_h - 2)], fill=(220, 220, 220), width=3)

    # Draw lines
    y = pad_y
    for i, line in enumerate(lines):
        bbox = draw.textbbox((0, 0), line, font=font)
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]
        x = pad_x  # left aligned; change to (width - w)//2 for centered
        draw.text((x, y), line, fill=fg, font=font)
        y += h + int(pad_y * 0.4)

    return header

In [21]:
# -----------------------------
# Load PNG images
# -----------------------------
img1 = Image.open(png_files[0]).convert("RGB")
img2 = Image.open(png_files[1]).convert("RGB")

# -----------------------------
# Convert df.head() → image
# -----------------------------
def df_to_image(df, title="", dpi=300):
    df = df.astype(str)

    nrows, ncols = df.shape

    # -----------------------------
    # Compute column widths from content length
    # -----------------------------
    max_lens = [
        max(
            df[col].str.len().max(),
            len(col)
        )
        for col in df.columns
    ]

    # normalize widths
    total = sum(max_lens)
    col_widths = [l / total for l in max_lens]

    # scale figure size based on total characters
    fig_w = max(16, total * 0.20)
    fig_h = max(6, 1.2 * (nrows + 1))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
    ax.axis("off")

    if title:
        ax.set_title(
            title,
            fontsize=20,
            pad=20,
            fontweight="bold"
        )

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        loc="center",
        cellLoc="left",
        colWidths=col_widths   # <-- KEY FIX
    )

    table.auto_set_font_size(False)
    table.set_fontsize(20)
    table.scale(1, 2.0)   # don't scale width blindly

    # header styling
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_fontsize(22)
            cell.set_text_props(weight='bold')

    tmp = RUN_DIR / "__tmp_df.png"
    fig.savefig(tmp, bbox_inches="tight", dpi=dpi)
    plt.close(fig)

    im = Image.open(tmp).convert("RGB")
    tmp.unlink()

    return im



# Convert to images
df_img1 = df_to_image(df1, title=Path(xlsx_files[0]).name, dpi=config['dpi'])
df_img2 = df_to_image(df2, title=Path(xlsx_files[1]).name, dpi=config['dpi'])

# -----------------------------
# Resize all to same width
# -----------------------------
def resize_to_width(im, w):
    h = int(im.size[1] * w / im.size[0])
    return im.resize((w, h), Image.LANCZOS)

panel_w = min(
    img1.size[0], img2.size[0],
    df_img1.size[0], df_img2.size[0]
)

img1 = resize_to_width(img1, panel_w)
img2 = resize_to_width(img2, panel_w)
df_img1 = resize_to_width(df_img1, panel_w*2)
df_img2 = resize_to_width(df_img2, panel_w*2)

# -----------------------------
# Create composite (2x2 grid)
# -----------------------------
gap = 40
pad = 60
bg = (255, 255, 255)

# Header (config banner)
header = make_header_image(config, width=pad*2 + panel_w*2 + gap, dpi=config["dpi"])

row2_h = max(df_img1.size[1], df_img2.size[1])
row1_h = max(img1.size[1], img2.size[1])

canvas_w = pad*2 + panel_w*2 + gap
canvas_h = (
    header.size[1]              
    + pad*2
    + row1_h
    + gap
    + df_img1.size[1]
    + gap
    + df_img2.size[1]
)


canvas = Image.new("RGB", (canvas_w, canvas_h), bg)

# -----------------------------
# Paste images
# -----------------------------
y0 = header.size[1]

# Header
canvas.paste(header, (0, 0))

# Top row (plots side by side)
canvas.paste(img1, (pad, y0 + pad))
canvas.paste(img2, (pad + panel_w + gap, y0 + pad))

# First dataframe (full width, centered under plots)
y_df1 = y0 + pad + row1_h + gap
canvas.paste(df_img1, (pad, y_df1))

# Second dataframe stacked below
y_df2 = y_df1 + df_img1.size[1] + gap
canvas.paste(df_img2, (pad, y_df2))
# -----------------------------
# Save
# -----------------------------
out_path = RUN_DIR / f"{run_name}_summary.png"

out_path.parent.mkdir(exist_ok=True)
canvas.save(out_path, dpi=(config["dpi"], config["dpi"]))

# Save summary to parent runs folder for quick comparisons
canvas.save(
    Path("runs") / f"{run_name}_summary.png",
    dpi=(config["dpi"], config["dpi"])
)

print(f"Saved composite image to: {out_path}")

Saved composite image to: runs\FLAVOR_MATCH_rd-true_meth-svd_dim-100_idf-false_2026-03-05_12-04\FLAVOR_MATCH_rd-true_meth-svd_dim-100_idf-false_2026-03-05_12-04_summary.png


# Searching

In [22]:
# kw = "iso"
# flavor_db.loc[flavor_db['Nom scientifique'].str.contains(kw)]